# Deriving $\delta^i{}_i = 3$ and $\delta^i{}_j\,\delta^i{}_j = 3$

This notebook demonstrates **derivations** — structured sequences of rewriting steps
applied to a tensor expression.  We prove two elementary identities for the Kronecker
delta in 3D oblique space using three uniform steps:

1. **`unroll_sums`** — expand each explicit sum over a concrete index space into a binary sum tree.
2. **`eval_delta_concrete`** — replace $\delta^a{}_b$ with concrete indices by $1$ (when $a = b$) or $0$.
3. **`fold_arithmetic`** — constant-fold the resulting scalar expression to a rational number.

In [ ]:
import tender
import tender.derivation as td
from tender import Level, Realm
from IPython.display import Math, display

sp = tender.space_3d  # index space {1, 2, 3}

## Helper: display a derivation history

Each step is shown on its own line as $= \text{expr}$.

In [ ]:
def show_derivation(drv, labels=None):
    """Display each expression in a derivation history as an align* block."""
    lines = []
    for k, expr in enumerate(drv.history):
        label = ""
        if labels and k > 0 and k - 1 < len(labels):
            label = rf"\quad\text{{\small {labels[k-1]}}}"
        lines.append(f"  &= {expr.latex()}{label}")
    body = " \\\\".join(lines)
    display(Math(rf"\begin{{align*}}{body}\end{{align*}}"))

STEP_LABELS = [
    "unroll sum",
    r"evaluate $\delta$ on concrete indices",
    "fold arithmetic",
]

## 1. Trace: $\displaystyle\sum_i \delta^i{}_i = 3$

Build the expression $\sum_i \delta^i{}_i$ and apply the three steps.

In [ ]:
ctx1 = tender.Context()
i = ctx1.alloc_index()

trace_expr = tender.explicit_sum(
    i,
    tender.delta(Realm.Oblique, sp, Level.Upper, Level.Lower, i, i, ctx=ctx1),
    ctx=ctx1,
)

display(Math(trace_expr.latex()))

In [ ]:
trace_drv = td.Derivation(trace_expr)
trace_drv.step(td.unroll_sums).step(td.eval_delta_concrete).step(td.fold_arithmetic)

show_derivation(trace_drv, STEP_LABELS)

## 2. Self-contraction: $\displaystyle\sum_i\sum_j \delta^i{}_j\,\delta^i{}_j = 3$

Each off-diagonal pair $(i \ne j)$ contributes $0 \cdot 0 = 0$; the three
diagonal pairs $(i = j)$ each contribute $1 \cdot 1 = 1$, giving $3$ in total.

In [ ]:
ctx2 = tender.Context()
i2 = ctx2.alloc_index()
j2 = ctx2.alloc_index()

d1 = tender.delta(Realm.Oblique, sp, Level.Upper, Level.Lower, i2, j2, ctx=ctx2)
d2 = tender.delta(Realm.Oblique, sp, Level.Upper, Level.Lower, i2, j2, ctx=ctx2)
contract_expr = tender.explicit_sum(
    i2,
    tender.explicit_sum(j2, d1 * d2, ctx=ctx2),
    ctx=ctx2,
)

display(Math(contract_expr.latex()))

In [ ]:
contract_drv = td.Derivation(contract_expr)
contract_drv.step(td.unroll_sums).step(td.eval_delta_concrete).step(td.fold_arithmetic)

show_derivation(contract_drv, STEP_LABELS)

## 3. Inspecting the history

`Derivation.history` is a plain list of expressions — one per step plus the initial.
You can access any intermediate result directly.

In [ ]:
print(f"Number of history entries: {len(contract_drv.history)}")
print()
for k, expr in enumerate(contract_drv.history):
    label = (["initial"] + STEP_LABELS)[k]
    print(f"  [{label}]")
    display(Math(expr.latex()))

## 4. Custom steps

Any callable `(Expr) -> Expr` works as a step.  Here we add a no-op
annotation step to illustrate the extension point.

In [ ]:
def identity_step(expr):
    """Return the expression unchanged (useful as a placeholder)."""
    return expr

ctx3 = tender.Context()
i3 = ctx3.alloc_index()
simple_expr = tender.explicit_sum(
    i3,
    tender.delta(Realm.Oblique, sp, Level.Upper, Level.Lower, i3, i3, ctx=ctx3),
    ctx=ctx3,
)

drv3 = td.Derivation(simple_expr)
drv3.step(identity_step).step(td.unroll_sums).step(td.eval_delta_concrete).step(td.fold_arithmetic)

print(f"Steps applied: {len(drv3.history) - 1}")
display(Math(drv3.current.latex()))